# Clase 12 — Multicolinealidad, selección de variables y validación

**Curso:** Estadística Aplicada con Python · **Bloque 4:** Modelos y experimentos
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/12_multicolinealidad_seleccion_validacion.md`

## Qué vas a ver en esta clase

- **Multicolinealidad** y **VIF**: por qué dos variables muy parecidas inflan los errores estándar y vuelven inestables los coeficientes.
- **Selección de variables** con criterio: dominio, significación, **AIC y BIC** (y el R² corregido en modelos lineales).
- El peligro de la **selección por búsqueda**: elegir entre 30 variables de puro ruido.
- **Sobreajuste** y validación **hold-out**, con NumPy y pandas.
- **Validación cruzada k-fold** a mano y comparación con un único hold-out.
- **Clases desbalanceadas**: exactitud frente a la predicción trivial, **matriz de confusión**, precisión y recall.
- **Data leakage**: seleccionar variables con datos de prueba.

## Requisitos

- Clase 10 (OLS, R² corregido, AIC) y clase 11 (modelo logístico, log-odds, odds ratio).
- Librerías: `numpy`, `pandas`, `matplotlib`, `statsmodels` (`pip install numpy pandas matplotlib statsmodels`; ya vienen instaladas en Google Colab).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Antes de cada experimento escribe qué esperas que ocurra. Los datos de abandono de clientes (*churn*) son sintéticos: un *ejemplo ilustrativo* generado con coeficientes verdaderos conocidos, para poder comparar la estimación con la verdad.

---
## Celda 0 — Setup

Importamos las librerías, fijamos la **semilla** y definimos el generador de clientes de la clase 11 (con sus **coeficientes verdaderos**, `VERDAD`), las métricas que calcularemos a mano (`log_loss`, `exactitud`, `matriz_confusion`) y la validación cruzada `kfold_log_loss`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

# Coeficientes VERDADEROS con los que generamos los datos (solo los conocemos porque son sintéticos)
VERDAD = {"Intercept": -1.8, "antiguedad": -0.03, "gasto": 0.03, "datos_gb": 0.0, "tickets": 0.45, "plan_anual": -0.9}

def generar_clientes(n, rng):
    """Genera n clientes sintéticos (ejemplo ilustrativo) con abandono según VERDAD."""
    antiguedad = np.clip(rng.gamma(2.0, 14, n), 1, 72).round()
    gasto = np.clip(30 + 0.25 * antiguedad + rng.normal(0, 10, n), 10, None).round(1)
    datos_gb = np.clip(0.09 * gasto + rng.normal(0, 0.25, n), 0.1, None).round(2)   # muy correlacionada con gasto
    tickets = rng.poisson(np.exp(0.008 * (gasto - 40)))
    plan_anual = rng.binomial(1, 0.35, n)
    eta = (VERDAD["Intercept"] + VERDAD["antiguedad"] * antiguedad + VERDAD["gasto"] * gasto
           + VERDAD["tickets"] * tickets + VERDAD["plan_anual"] * plan_anual)
    abandona = rng.binomial(1, 1 / (1 + np.exp(-eta)))
    return pd.DataFrame({"abandona": abandona, "antiguedad": antiguedad, "gasto": gasto,
                         "datos_gb": datos_gb, "tickets": tickets, "plan_anual": plan_anual})

def log_loss(y, p):
    """Pérdida logarítmica media (menor es mejor)."""
    y, p = np.asarray(y), np.asarray(p)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def exactitud(y, p, umbral=0.5):
    """Proporción de aciertos al clasificar con un umbral."""
    return float(np.mean((np.asarray(p) > umbral) == np.asarray(y)))

def matriz_confusion(y, p, umbral=0.5):
    """Devuelve (VP, FP, FN, VN) al clasificar 'abandona' si p > umbral."""
    y, pred = np.asarray(y), np.asarray(p) > umbral
    return (int(((pred == 1) & (y == 1)).sum()), int(((pred == 1) & (y == 0)).sum()),
            int(((pred == 0) & (y == 1)).sum()), int(((pred == 0) & (y == 0)).sum()))

def kfold_log_loss(formula, d, k, rng):
    """Validación cruzada k-fold a mano: devuelve el log-loss de cada bloque."""
    orden = rng.permutation(len(d))
    folds = np.array_split(orden, k)
    perdidas = []
    for i in range(k):
        test = d.iloc[folds[i]]
        train = d.iloc[np.concatenate([folds[j] for j in range(k) if j != i])]
        m = smf.logit(formula, data=train).fit(disp=0)
        perdidas.append(log_loss(test["abandona"], m.predict(test)))
    return np.array(perdidas)

print("Librerías y funciones cargadas correctamente.")

---
## 1. Los datos: abandono de clientes (*churn*)

Los mismos 2 000 clientes de la clase 11. La respuesta `abandona` vale 1 si el cliente se da de baja. Las explicativas: antigüedad (meses), gasto mensual (€), tickets de soporte, plan anual (0/1) y GB de datos (`datos_gb`), esta última **muy correlacionada con el gasto** y **sin efecto propio** (coeficiente verdadero 0).

In [ ]:
df = generar_clientes(2000, rng)
idx = rng.permutation(len(df))     # orden aleatorio de filas para el hold-out (sección 7)

display(df.head())
print("Tasa de abandono:", round(df["abandona"].mean() * 100, 1), "%")
print("Correlación gasto - datos_gb:", round(df["gasto"].corr(df["datos_gb"]), 2))

---
## 2. Multicolinealidad: qué le hace a un coeficiente

Ajustamos el modelo logístico de la clase 11 **sin** y **con** `datos_gb` y comparamos el coeficiente y el error estándar de `gasto`.

In [ ]:
FORM = "abandona ~ antiguedad + gasto + tickets + plan_anual"
m_sin = smf.logit(FORM, data=df).fit(disp=0)
m_con = smf.logit(FORM + " + datos_gb", data=df).fit(disp=0)

tabla = pd.DataFrame({
    "coef. de gasto": [m_sin.params["gasto"], m_con.params["gasto"]],
    "error estándar": [m_sin.bse["gasto"], m_con.bse["gasto"]],
    "p-value": [m_sin.pvalues["gasto"], m_con.pvalues["gasto"]]},
    index=["sin datos_gb", "con datos_gb"])
display(tabla.round(4))
print("Cociente de errores estándar (con / sin):", round(m_con.bse["gasto"] / m_sin.bse["gasto"], 1))
print("Coeficiente verdadero de gasto:", VERDAD["gasto"])

**¿Qué observas?** Con `datos_gb` el error estándar de `gasto` se multiplica casi por 4 y su p-value deja de ser significativo, aunque el efecto es real. El coeficiente de `datos_gb`, cuyo valor verdadero es 0, sale positivo y también con error grande: el modelo no sabe repartir el efecto entre dos variables que se mueven casi juntas.

### Experimento guiado 1: la inestabilidad de los coeficientes

Repetimos el estudio **200 veces** con muestras nuevas de 500 clientes, ajustando el modelo con y sin `datos_gb`, y miramos cuánto varía el coeficiente de `gasto` de una muestra a otra. **Antes de ejecutar, predice:** ¿en qué modelo variará más? ¿Y con qué frecuencia saldrá con el signo equivocado?

In [ ]:
rng_col = np.random.default_rng(123)
coef_sin, coef_con = [], []
for _ in range(200):
    d = generar_clientes(500, rng_col)
    coef_sin.append(smf.logit(FORM, data=d).fit(disp=0).params["gasto"])
    coef_con.append(smf.logit(FORM + " + datos_gb", data=d).fit(disp=0).params["gasto"])

print("Coeficiente verdadero de gasto:", VERDAD["gasto"])
print("Sin datos_gb → media:", round(np.mean(coef_sin), 4), " desv. típica:", round(np.std(coef_sin, ddof=1), 4))
print("Con datos_gb → media:", round(np.mean(coef_con), 4), " desv. típica:", round(np.std(coef_con, ddof=1), 4))
print("% de muestras con signo equivocado (con datos_gb):", round(np.mean(np.array(coef_con) < 0) * 100, 1))

plt.hist(coef_sin, bins=25, alpha=0.6, label="sin datos_gb")
plt.hist(coef_con, bins=25, alpha=0.6, label="con datos_gb")
plt.axvline(VERDAD["gasto"], color="black", linestyle="--", label="verdad")
plt.title("Coeficiente de gasto en 200 muestras")
plt.legend()
plt.show()

**¿Qué observas?** Las dos medias están cerca de la verdad (0,03), pero con `datos_gb` la dispersión es unas cuatro veces mayor y en más de una de cada cinco muestras el coeficiente sale **negativo**. El modelo es correcto en promedio y poco fiable en cada muestra. Es lo que significa "coeficientes inestables".

---
## 3. Detectar la multicolinealidad: el VIF

`VIF = 1 / (1 − R²)`, donde R² es el de explicar esa variable con todas las demás. Lo calculamos de tres maneras: a mano con la correlación, con la regresión auxiliar y con `variance_inflation_factor`.

In [ ]:
columnas = ["antiguedad", "gasto", "datos_gb", "tickets", "plan_anual"]

r = df["gasto"].corr(df["datos_gb"])
print("VIF a mano con solo dos variables, 1/(1-r²) con r = 0,97:", round(1 / (1 - 0.97 ** 2), 1))

aux = smf.ols("gasto ~ antiguedad + datos_gb + tickets + plan_anual", data=df).fit()
print("R² de la regresión auxiliar de gasto:", round(aux.rsquared, 3), "→ VIF =", round(1 / (1 - aux.rsquared), 2))

def tabla_vif(datos, cols):
    Xm = sm.add_constant(datos[cols])
    return pd.Series({c: variance_inflation_factor(Xm.values, i) for i, c in enumerate(Xm.columns) if c != "const"})

display(pd.DataFrame({"con datos_gb": tabla_vif(df, columnas)}).round(2))
display(pd.DataFrame({"sin datos_gb": tabla_vif(df, ["antiguedad", "gasto", "tickets", "plan_anual"])}).round(2))

**¿Qué observas?** Los dos VIF altos (≈ 17) son justo el par `gasto` – `datos_gb`; los demás están cerca de 1. Al quitar `datos_gb` todos vuelven a valores ≈ 1. Recuerda: los umbrales (5, 10) son convenciones; la decisión de qué variable quitar es de **dominio** (aquí, el gasto es lo que la empresa factura y controla).

---
## 4. Elegir variables: R² corregido en modelos lineales

En la clase 10 vimos el R² corregido. Comprobamos que el R² normal **sube siempre** al añadir variables inútiles y que R² corregido, AIC y BIC lo detectan. Modelo lineal con 300 clientes y respuesta `gasto`; añadimos 3 columnas de ruido puro.

In [ ]:
rl = np.random.default_rng(31)
dl = generar_clientes(300, rl)
for j in range(1, 4):
    dl[f"nz{j}"] = rl.normal(size=300)

cand_lin = {"solo antiguedad": "gasto ~ antiguedad",
            "+ 3 variables de ruido": "gasto ~ antiguedad + nz1 + nz2 + nz3"}
filas = []
for nombre, f in cand_lin.items():
    m = smf.ols(f, data=dl).fit()
    filas.append([m.rsquared, m.rsquared_adj, m.aic, m.bic])
display(pd.DataFrame(filas, index=list(cand_lin), columns=["R²", "R² corregido", "AIC", "BIC"]).round(4))

**¿Qué observas?** El R² **sube** con las tres variables de ruido (aunque no explican nada); el R² corregido **baja** y AIC y BIC **empeoran** (suben). Cualquiera de las tres medidas corregidas detecta que las variables sobran; el R² normal, no.

---
## 5. AIC y BIC en el modelo logístico

`AIC = 2k − 2·LL` y `BIC = k·ln(n) − 2·LL`, con `k` parámetros (contando la constante) y `LL` la log-verosimilitud. Comprobamos la fórmula a mano con el modelo M3 y comparamos los cinco candidatos.

In [ ]:
m3 = smf.logit(FORM, data=df).fit(disp=0)
k, n, LL = len(m3.params), int(m3.nobs), m3.llf
print("k =", k, "  n =", n, "  LL =", round(LL, 1))
print("AIC a mano:", round(2 * k - 2 * LL, 1), "  statsmodels:", round(m3.aic, 1))
print("BIC a mano:", round(k * np.log(n) - 2 * LL, 1), "  statsmodels:", round(m3.bic, 1))

candidatos = {
    "M0: antiguedad": "abandona ~ antiguedad",
    "M1: + tickets": "abandona ~ antiguedad + tickets",
    "M2: + plan_anual": "abandona ~ antiguedad + tickets + plan_anual",
    "M3: + gasto": "abandona ~ antiguedad + tickets + plan_anual + gasto",
    "M4: + datos_gb": "abandona ~ antiguedad + tickets + plan_anual + gasto + datos_gb"}
ajustes = {nombre: smf.logit(f, data=df).fit(disp=0) for nombre, f in candidatos.items()}
tabla_ic = pd.DataFrame({"k": [len(m.params) for m in ajustes.values()],
                         "AIC": [m.aic for m in ajustes.values()],
                         "BIC": [m.bic for m in ajustes.values()]}, index=list(ajustes))
display(tabla_ic.round(1))
print("Menor AIC:", tabla_ic["AIC"].idxmin(), "  |  Menor BIC:", tabla_ic["BIC"].idxmin())

**¿Qué observas?** El cálculo a mano coincide con `statsmodels`. Los dos criterios bajan hasta M3 y suben en M4: `datos_gb` no compensa su parámetro extra. Con 2 000 clientes AIC y BIC coinciden; el BIC castiga cada parámetro con ln(2000) ≈ 7,6 frente a 2 del AIC. ¿Coinciden siempre? Lo vemos ahora.

### Experimento guiado 2: AIC frente a BIC con muestras de 500 clientes

Primero una muestra concreta y después 200 muestras nuevas de 500 clientes: ¿qué modelo elige cada criterio? El modelo verdadero es M3. **Antes de ejecutar, predice:** ¿quién elegirá más veces un modelo demasiado grande? ¿Y demasiado pequeño?

In [ ]:
d500 = generar_clientes(500, np.random.default_rng(100))
aj500 = {nombre: smf.logit(f, data=d500).fit(disp=0) for nombre, f in candidatos.items() if nombre[:2] in ("M2", "M3", "M4")}
display(pd.DataFrame({"AIC": [m.aic for m in aj500.values()], "BIC": [m.bic for m in aj500.values()]}, index=list(aj500)).round(1))

elige_aic, elige_bic = np.zeros(5, int), np.zeros(5, int)
r_sim = np.random.default_rng(2025)
for _ in range(200):
    d = generar_clientes(500, r_sim)
    aj = [smf.logit(f, data=d).fit(disp=0) for f in candidatos.values()]
    elige_aic[np.argmin([m.aic for m in aj])] += 1
    elige_bic[np.argmin([m.bic for m in aj])] += 1
res_ic = pd.DataFrame({"AIC (% de muestras)": elige_aic / 2, "BIC (% de muestras)": elige_bic / 2},
                      index=["M0", "M1", "M2", "M3", "M4"]).T
display(res_ic)

**¿Qué observas?** En la muestra concreta los AIC de M2 y M3 casi coinciden (diferencia < 2: indistinguibles) y el BIC prefiere el más simple. En las 200 muestras, el AIC acierta el modelo verdadero (M3) con más frecuencia pero a veces mantiene la variable redundante (M4); el BIC casi nunca elige M4 pero con 500 clientes se queda a menudo en M2 y **omite `gasto`**, cuyo efecto es real aunque modesto. No hay ganador universal: AIC orientado a predecir, BIC a la parsimonia.

---
## 6. El peligro de la selección por búsqueda

### Experimento guiado 3: selección sobre puro ruido

200 repeticiones. En cada una: 200 clientes, una respuesta 0/1 **generada al azar** y **30 variables de ruido**. Ajustamos el modelo con las 30, nos quedamos con las de p < 0,05, reajustamos y miramos el mejor p-value. **Antes de ejecutar, predice:** si todo es ruido, ¿cuántas repeticiones acabarán con un modelo "convincente"?

In [ ]:
rng_sw = np.random.default_rng(11)
n_elegidas, p_min_final = [], []
for _ in range(200):
    Xr = pd.DataFrame(rng_sw.normal(size=(200, 30)), columns=[f"x{j}" for j in range(30)])
    yr = rng_sw.binomial(1, 0.3, 200)
    completo = sm.Logit(yr, sm.add_constant(Xr)).fit(disp=0, maxiter=200)
    elegidas = [c for c in Xr.columns if completo.pvalues[c] < 0.05]
    n_elegidas.append(len(elegidas))
    if elegidas:
        final = sm.Logit(yr, sm.add_constant(Xr[elegidas])).fit(disp=0)
        p_min_final.append(final.pvalues[elegidas].min())

n_elegidas, p_min_final = np.array(n_elegidas), np.array(p_min_final)
print("Variables elegidas por repetición (media):", round(n_elegidas.mean(), 2), "(por azar se esperan 1,5)")
print("% de repeticiones con al menos una variable elegida:", round((n_elegidas > 0).mean() * 100, 1))
print("De esas, % con al menos una p < 0,05 en el modelo final:", round((p_min_final < 0.05).mean() * 100, 1))
print("Mediana del p-value mínimo del modelo final:", round(np.median(p_min_final), 3))

**¿Qué observas?** En la mayoría de las repeticiones se "descubren" variables útiles que **no existen**, con p-values finales pequeños. Los p-values del modelo reajustado son optimistas porque las variables se eligieron precisamente por parecer buenas. Por eso: candidatas definidas con criterio de dominio antes de mirar los datos y evaluación con datos no usados para elegir.

---
## 7. Sobreajuste y validación hold-out

Separamos 1 400 clientes de entrenamiento y 600 de prueba (con el orden aleatorio `idx` de la sección 1), estimamos en el entrenamiento y evaluamos **a mano** con log-loss y exactitud. Como referencia usamos la predicción constante (la tasa de abandono de entrenamiento).

In [ ]:
train = df.iloc[idx[:1400]]
test = df.iloc[idx[1400:]]
m_train = smf.logit(FORM, data=train).fit(disp=0)

p_tr, p_te = m_train.predict(train), m_train.predict(test)
p_base = np.full(len(test), train["abandona"].mean())      # predicción constante

resultados = pd.DataFrame({
    "log-loss": [log_loss(train["abandona"], p_tr), log_loss(test["abandona"], p_te), log_loss(test["abandona"], p_base)],
    "exactitud": [exactitud(train["abandona"], p_tr), exactitud(test["abandona"], p_te), exactitud(test["abandona"], p_base)]},
    index=["modelo, entrenamiento", "modelo, prueba", "constante, prueba"])
display(resultados.round(3))
print("Exactitud de 'nadie abandona' en la prueba:", round((test["abandona"] == 0).mean(), 3))

**¿Qué observas?** Entrenamiento y prueba dan casi lo mismo: con 4 variables y 1 400 clientes no hay sobreajuste. Fíjate en la **exactitud**: el modelo apenas supera a "nadie abandona" (unos 0,77 frente a 0,765), aunque su log-loss es claramente mejor que el de la predicción constante. Con clases desbalanceadas la exactitud engaña (sección 9).

### Experimento guiado 4: añadir variables de ruido

30 repeticiones con **200 clientes de entrenamiento** y 1 000 de prueba, añadiendo `k` = 0, 5, 10, 20 y 30 variables de puro ruido a las 4 reales. **Antes de ejecutar, predice:** ¿qué le pasa al log-loss en entrenamiento y en prueba al aumentar `k`?

In [ ]:
def simular_sobreajuste(n_train, k, repeticiones=30, semilla=7):
    r = np.random.default_rng(semilla)
    perdidas = []
    for _ in range(repeticiones):
        d = generar_clientes(n_train + 1000, r)
        for j in range(k):
            d[f"nz{j + 1}"] = r.normal(size=len(d))
        tr, te = d.iloc[:n_train], d.iloc[n_train:]
        formula = FORM + "".join(f" + nz{j + 1}" for j in range(k))
        m = smf.logit(formula, data=tr).fit(disp=0, maxiter=200)
        perdidas.append((log_loss(tr["abandona"], m.predict(tr)), log_loss(te["abandona"], m.predict(te))))
    return np.mean(perdidas, axis=0)

ks = [0, 5, 10, 20, 30]
res_sobre = pd.DataFrame([simular_sobreajuste(200, k) for k in ks], index=ks,
                         columns=["log-loss entrenamiento", "log-loss prueba"])
res_sobre.index.name = "variables de ruido"
display(res_sobre.round(3))

res_sobre.plot(marker="o")
plt.axhline(log_loss(test["abandona"], p_base), color="gray", linestyle="--", label="predicción constante")
plt.title("El ajuste en entrenamiento mejora, la prueba empeora")
plt.ylabel("log-loss")
plt.legend()
plt.show()

**¿Qué observas?** Cada variable inútil hace **bajar** el log-loss de entrenamiento (parece que el modelo ajusta mejor) y **subir** el de prueba. Con muchas variables de ruido el modelo rinde peor con datos nuevos que la predicción constante (≈ 0,55). El hueco entre las dos curvas es el sobreajuste, y solo se ve si evaluamos con datos que el modelo no ha visto.

---
## 8. Validación cruzada k-fold, a mano

Con 300 clientes repartimos los datos en 5 bloques de 60. Cada bloque hace de prueba una vez y el modelo se estima con los otros cuatro. Usamos la función `kfold_log_loss` de la Celda 0 (léela: son solo diez líneas de NumPy y pandas).

In [ ]:
d300 = generar_clientes(300, np.random.default_rng(5))
perd = kfold_log_loss(FORM, d300, 5, np.random.default_rng(1))
print("Log-loss por bloque:", perd.round(3))
print("Media (estimación 5-fold):", round(perd.mean(), 3))

### Experimento guiado 5: k-fold frente a un único hold-out

Con los mismos 300 clientes repetimos **200 veces** cada procedimiento con particiones al azar distintas: (a) un hold-out 70/30 y (b) un 5-fold (media de los 5 bloques). **Antes de ejecutar, predice:** ¿tendrán la misma media? ¿Cuál variará más?

In [ ]:
ho, cv = [], []
for s in range(200):
    rr = np.random.default_rng(1000 + s)
    o = rr.permutation(300)
    a, b = d300.iloc[o[:210]], d300.iloc[o[210:]]
    m = smf.logit(FORM, data=a).fit(disp=0)
    ho.append(log_loss(b["abandona"], m.predict(b)))
    cv.append(kfold_log_loss(FORM, d300, 5, rr).mean())
ho, cv = np.array(ho), np.array(cv)

comparacion = pd.DataFrame({"media": [ho.mean(), cv.mean()], "desv. típica": [ho.std(ddof=1), cv.std(ddof=1)],
                            "mínimo": [ho.min(), cv.min()], "máximo": [ho.max(), cv.max()]},
                           index=["hold-out 70/30", "5-fold"])
display(comparacion.round(3))
print("Cociente de desviaciones típicas (hold-out / 5-fold):", round(ho.std(ddof=1) / cv.std(ddof=1), 1))

plt.hist(ho, bins=25, alpha=0.6, label="hold-out 70/30")
plt.hist(cv, bins=25, alpha=0.6, label="5-fold")
plt.title("Log-loss estimado en 200 particiones distintas")
plt.legend()
plt.show()

**¿Qué observas?** Las medias son casi iguales, pero el hold-out cambia enormemente con la partición (rango de unos 0,35 puntos de log-loss): con la partición que te toque podrías concluir que el modelo es excelente o mediocre. El 5-fold, que prueba con todos los datos, es unas ocho veces más estable, a cambio de ajustar el modelo cinco veces.

---
## 9. Clases desbalanceadas: matriz de confusión, precisión y recall

Con el modelo `m_train` y los 600 clientes de prueba de la sección 7 clasificamos "abandona" si `p > umbral`, contamos los cuatro resultados posibles y calculamos exactitud, precisión y recall **a mano**.

In [ ]:
p_test, y_test = m_train.predict(test).values, test["abandona"].values
vp, fp, fn, vn = matriz_confusion(y_test, p_test, 0.5)

display(pd.DataFrame([[vp, fp], [fn, vn]],
                     index=["predicho: abandona", "predicho: no abandona"],
                     columns=["real: abandona", "real: no abandona"]))
print("Exactitud = (VP + VN) / total =", round((vp + vn) / len(y_test), 3))
print("Precisión = VP / (VP + FP) =", round(vp / (vp + fp), 3))
print("Recall = VP / (VP + FN) =", round(vp / (vp + fn), 3))
print("'Nadie abandona': exactitud", round(np.mean(y_test == 0), 3), "  recall = 0")

**¿Qué observas?** La exactitud (0,77) parece buena, pero "nadie abandona" ya acierta el 76,5 % con recall 0. Nuestro modelo, con el umbral por defecto, detecta solo una minoría de los que se van. ¿Y si movemos el umbral?

In [ ]:
filas = []
for umbral in (0.5, 0.3, 0.2):
    vp, fp, fn, vn = matriz_confusion(y_test, p_test, umbral)
    filas.append([umbral, vp, fp, fn, vn, (vp + vn) / len(y_test), vp / (vp + fp), vp / (vp + fn)])
display(pd.DataFrame(filas, columns=["umbral", "VP", "FP", "FN", "VN", "exactitud", "precisión", "recall"]).round(3))

**¿Qué observas?** Al bajar el umbral **sube el recall** (detectamos más abandonos) y **baja la precisión** (más falsas alarmas); la exactitud incluso empeora. No hay umbral "correcto" en abstracto: depende del coste de perder un cliente frente al de una oferta de retención innecesaria. (Curvas ROC/AUC: curso de Machine Learning.)

---
## 10. Data leakage: seleccionar variables con los datos de prueba

### Experimento guiado 6: la fuga de información

200 repeticiones. En cada una: 100 filas, 40 variables de **puro ruido** y una respuesta 0/1 al azar (50 %): ningún modelo honesto puede acertar más de ≈ 50 %. Elegimos las 5 variables más correlacionadas con la respuesta y medimos la exactitud en el 30 % de prueba de dos formas: **(a) con fuga**, eligiendo con las 100 filas y separando después; **(b) correcta**, eligiendo solo con el 70 % de entrenamiento. **Antes de ejecutar, predice** la exactitud media de cada procedimiento.

In [ ]:
def elegir_top(Xm, y, filas, k=5):
    corr = np.abs([np.corrcoef(Xm[filas, j], y[filas])[0, 1] for j in range(Xm.shape[1])])
    return np.argsort(corr)[-k:]

rl2 = np.random.default_rng(3)
acc_fuga, acc_ok = [], []
for _ in range(200):
    Xm = rl2.normal(size=(100, 40))
    y = rl2.binomial(1, 0.5, 100)
    orden = rl2.permutation(100)
    tr_i, te_i = orden[:70], orden[70:]
    for con_fuga, lista in ((True, acc_fuga), (False, acc_ok)):
        cols = elegir_top(Xm, y, np.arange(100)) if con_fuga else elegir_top(Xm, y, tr_i)
        m = sm.Logit(y[tr_i], sm.add_constant(Xm[tr_i][:, cols])).fit(disp=0)
        pr = m.predict(sm.add_constant(Xm[te_i][:, cols], has_constant="add"))
        lista.append(((pr > 0.5) == y[te_i]).mean() * 100)

print("Exactitud media en prueba, CON fuga (%):", round(np.mean(acc_fuga), 1))
print("Exactitud media en prueba, procedimiento correcto (%):", round(np.mean(acc_ok), 1))
print("Error estándar de la media (fuga):", round(np.std(acc_fuga, ddof=1) / np.sqrt(200), 1))

**¿Qué observas?** Con fuga, un modelo construido sobre **puro ruido** parece acertar más del 60 %; con el procedimiento correcto obtiene ≈ 50 %, lo que corresponde al azar. La fuga no da ningún aviso: solo resultados demasiado buenos. Regla: todo paso que mire la respuesta se hace **solo con el entrenamiento** (y dentro de cada bloque en la validación cruzada).

---
## Resumen de la clase

- La **multicolinealidad** infla errores estándar y desestabiliza coeficientes sin dañar las predicciones; se detecta con el **VIF** y se resuelve quitando o combinando variables, con criterio de dominio.
- Se eligen variables con dominio, significación y **AIC/BIC** (o R² corregido en modelos lineales); el BIC castiga más la complejidad y los dos pueden discrepar.
- Buscar entre muchas variables da p-values optimistas: define candidatas antes de mirar y **evalúa con datos no usados**.
- El **sobreajuste** se ve al comparar entrenamiento y prueba; un solo hold-out es inestable, el **k-fold** es más estable.
- Con clases desbalanceadas la exactitud engaña: usa la **matriz de confusión**, la precisión y el recall.
- El **data leakage** (elegir variables con los datos de prueba) hace parecer bueno un modelo inútil.

**Siguiente paso:** el notebook de práctica de esta clase y después la clase 13: Test A/B y diseño de experimentos.